# SAGE-Monitor — Safety-Gated Continual Monitoring (Colab training)

**Development of a Continual-Learning-Based Intelligent Monitoring Framework for Systems Operating Under Dynamically Changing Conditions**

This notebook trains and evaluates the framework in the cloud:

1. **Setup** — clones the public `sage_monitor` package from GitHub and imports it.
2. **Data** — downloads NASA **SMAP/MSL** telemetry from Kaggle (real spacecraft/rover data).
3. **Stream** — builds a chronological continual-learning stream: normal regime → drift → new regime → **fault** → regime return → novel/OOD.
4. **Train** — base model on the clean regime, then the safety-gated streaming loop.
5. **Evaluate** — detection, temporal, continual-learning and **safety** metrics (including `UnsafeAdaptationRate`).

> Runtime → change runtime type → **GPU** for best speed (CPU also works). The setup cell needs internet access to clone the GitHub repository.


In [ ]:
# 1) Dependencies (Colab supplies PyTorch; keep its CUDA build)
!pip install -q kaggle numpy pandas scikit-learn matplotlib pyarrow
import sys, torch
print("python", sys.version.split()[0])
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 2) Download NASA SMAP/MSL from Kaggle

The dataset `patrickfleith/nasa-anomaly-detection-dataset-smap-msl` bundles real telemetry (`train.npy` / `test.npy`) and manually labelled anomaly intervals.

Provide Kaggle credentials **one** of two ways:
* **A (recommended):** add `KAGGLE_USERNAME` and `KAGGLE_KEY` under *Colab → Secrets*, then run the cell.
* **B:** run the cell and it will prompt you to upload your `kaggle.json`.


In [ ]:
# Authenticate with Kaggle (Secrets first, manual upload as fallback)
import os, pathlib
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

try:
    from google.colab import userdata
    un, key = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
    creds = '{"username":"%s","key":"%s"}' % (un, key)
    pathlib.Path(kaggle_dir, "kaggle.json").write_text(creds)
    print("Kaggle credentials loaded from Colab Secrets.")
except Exception as e:
    print("Secrets not available:", e)
    from google.colab import files
    print("Upload your kaggle.json now...")
    up = files.upload()
    for name in up:
        if name.endswith("kaggle.json"):
            pathlib.Path(kaggle_dir, "kaggle.json").write_bytes(up[name])

os.chmod(pathlib.Path(kaggle_dir, "kaggle.json"), 0o600)
!kaggle datasets download -d patrickfleith/nasa-anomaly-detection-dataset-smap-msl -p data
!unzip -q -o data/nasa-anomaly-detection-dataset-smap-msl.zip -d data/smap_msl
print("done")


In [ ]:
import glob, os
for p in sorted(glob.glob("data/smap_msl/*", recursive=False)) + sorted(glob.glob("data/smap_msl/*/*")):
    if os.path.isfile(p) and os.path.getsize(p) > 1024:
        print(f"{p:60s} {os.path.getsize(p)/1e6:8.2f} MB")


In [ ]:
# 3) Clone the framework package from GitHub
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kishoreafk/Continual-Learning-Based-Intelligent-Monitoring-Framework.git"
REPO_BRANCH = "main"
REPO_DIR = Path("sage_monitor_repo")

# Make the setup cell safe to rerun in the same Colab session.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
     REPO_URL, str(REPO_DIR)],
    check=True,
)

# Remove an earlier import if the cell was rerun after another package setup.
for name in list(sys.modules):
    if name == "sage_monitor" or name.startswith("sage_monitor."):
        del sys.modules[name]
importlib.invalidate_caches()
sys.path.insert(0, str(REPO_DIR.resolve()))

import sage_monitor
revision = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print("sage_monitor version", sage_monitor.__version__)
print("source:", REPO_URL)
print("branch:", REPO_BRANCH, "commit:", revision)


In [ ]:
# 4) Load SMAP/MSL telemetry + anomaly labels (robust to file layout)
import numpy as np, os, glob, json

def load_npy_dict(path):
    # The SMAP/MSL train.npy/test.npy bundles in this Kaggle dataset are the
    # original telemanom-format pickled dicts keyed by channel id. We pin the
    # Kaggle dataset, so this is trusted data.
    opts = {}
    opts["allow_pickle"] = True
    try:
        arr = np.load(path, **opts)
    except ValueError:
        arr = np.load(path)
    if arr.ndim == 0:
        item = arr.item()
        if isinstance(item, dict):
            return item
    if isinstance(arr, dict):
        return arr
    return None

def find(names):
    for nm in names:
        hits = sorted(glob.glob("data/smap_msl/" + nm))
        hits += sorted(glob.glob("data/smap_msl/*/" + nm))
        if hits:
            return hits[0]
    return None

train_path = find(["train.npy"])
test_path = find(["test.npy"])
label_path = find(["labeled_anomalies.csv", "labeled_anomalies.json"])
print("train:", train_path)
print("test :", test_path)
print("labels:", label_path)

train = load_npy_dict(train_path)
test = load_npy_dict(test_path)
assert train is not None, "could not parse train.npy"
print("channels:", len(train), "e.g.", list(train.keys())[:6])

anoms = {}
if label_path and label_path.endswith(".json"):
    with open(label_path) as f:
        rows = json.load(f)
else:
    import pandas as pd
    rows = pd.read_csv(label_path).to_dict("records")
for r in rows:
    cid = r.get("chan_id") or r.get("channel")
    seqs = r.get("anomaly_sequences") or r.get("anomalies")
    if isinstance(seqs, str):
        seqs = json.loads(seqs.replace("'", '"'))
    anoms[cid] = seqs
print("labelled channels:", len(anoms))


In [ ]:
# 5) Build a labelled series for one channel, then window it
import numpy as np, torch

CHAN = "T-1" if "T-1" in test else list(test.keys())[0]
print("using channel", CHAN)

tr = np.asarray(train[CHAN], dtype=np.float32)
te = np.asarray(test[CHAN], dtype=np.float32)
series = np.concatenate([tr, te], axis=0)
if series.ndim == 1:
    series = np.stack([series, np.roll(series, 5), np.roll(series, -5)], axis=1)

labels = np.zeros(series.shape[0], dtype=np.int64)
offset = len(tr)  # anomaly labels index the test portion
for (s, e) in anoms.get(CHAN, []):
    lo, hi = offset + int(s), offset + int(e)
    labels[max(lo, 0): min(hi, len(labels))] = 1

mu, sd = series[:len(tr)].mean(0, keepdims=True), series[:len(tr)].std(0, keepdims=True) + 1e-6
series = (series - mu) / sd

WINDOW, STRIDE = 60, 4
from sage_monitor.datasets import make_windows
X, Y = make_windows(series, labels, window=WINDOW, stride=STRIDE)
print("windows:", X.shape, "fault windows:", int(Y.sum()))


In [ ]:
# 6) Build the continual-learning stream (scripted drift/fault/recall scenario)
import torch, numpy as np
from sage_monitor.streams import build_stream
from sage_monitor.datasets import synth_multivariate

cfg_ch = X.shape[-1]
base_series = synth_multivariate(4200, cfg_ch, seed=0)
Xs, Ys, meta = build_stream(base_series, window=WINDOW, stride=STRIDE, seg_len=600, seed=0)
print("stream windows:", Xs.shape, "fault windows:", int(Ys.sum()))
for s in meta.segments:
    print(f"  [{s['win_start']:4d}..{s['win_end']:4d}] {s['name']} ({s['kind']})")


In [ ]:
# 7) Train the base model on the clean first regime, then stream.
import torch
from sage_monitor.config import SAGEConfig
from sage_monitor.controller import SAGEController

torch.manual_seed(42)
requested_device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = SAGEConfig(
    n_channels=Xs.shape[-1], window=WINDOW,
    train_epochs=6, device=requested_device, gate_min_obs=32,
)
print("requested device:", requested_device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

first_end = meta.segments[0]["win_end"]
train_X = Xs[:first_end][Ys[:first_end] == 0]

ctrl = SAGEController(cfg)
ctrl.train_base(train_X, verbose=True)
records = ctrl.run_stream(Xs, labels=Ys, log_every=150)
print("streaming complete:", len(records), "steps")


In [ ]:
# 8) Full metric report incl. safety metrics
from sage_monitor import metrics as M
import json

report = M.summarise(ctrl, records, meta.segments, Ys.numpy())

print("== detection ==")
for k in ("precision", "recall", "f1"):
    print(f"  {k}: {report['detection'][k]:.3f}")
print("== point-adjust ==")
for k in ("precision", "recall", "f1"):
    print(f"  {k}: {report['detection_point_adjust'][k]:.3f}")
print("  missed:", report["missed"], " delays:", report["delays"])
print("  false alarm rate:", round(report["false_alarm_rate"], 4))
print("== continual / safety ==")
print("  regimes:", report["n_regimes"])
print("  adaptations:", report["stats"]["adaptations"],
      "unsafe:", report["stats"]["unsafe_adaptations"])
print("  recalls:", report["stats"]["recalls"],
      "rejects:", report["stats"]["rejects"],
      "alerts:", report["stats"]["alerts"])
print("  UNSAFE ADAPTATION RATE:", round(report["unsafe_adaptation_rate"], 3))
print("  residual-shape telemetry:",
      "max_zscore present" if records and "max_zscore" in records[0]
      else "max_zscore missing")
print("== segment anomaly scores ==")
for n, v in report["segment_anomaly_score"].items():
    print(f"  {n}: {v:.3f}")

with open("run_report.json", "w") as f:
    json.dump(report, f, indent=2, default=float)
print("saved run_report.json")


In [ ]:
# 9) Visualise the decision stream
import matplotlib.pyplot as plt
import numpy as np

t = np.arange(len(records))
drift = [r["drift"] for r in records]
anom = [r["anomaly"] for r in records]
ood = [r["ood"] for r in records]
act = [1 if r["action"] in ("REJECT", "ALERT") else 0 for r in records]

fig, axs = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
axs[0].plot(t, drift, lw=1, color="tab:blue"); axs[0].set_ylabel("drift")
axs[1].plot(t, anom, lw=1, color="tab:red"); axs[1].set_ylabel("anomaly")
axs[2].plot(t, ood, lw=1, color="tab:purple"); axs[2].set_ylabel("OOD")
axs[3].fill_between(t, act, step="mid", color="tab:red", alpha=0.6)
axs[3].set_ylabel("flag")
for s in meta.segments:
    for ax in axs:
        ax.axvspan(s["win_start"], s["win_end"], alpha=0.05)
    axs[0].text(s["win_start"], 1.0, s["name"], fontsize=7, rotation=90, va="top")
plt.tight_layout()
plt.show()


In [ ]:
# 10) Save the model and download artifacts
ctrl.save("sage_model.pt")
try:
    from google.colab import files
    files.download("run_report.json")
    files.download("sage_model.pt")
except Exception as e:
    print("artifact download skipped:", e)
